# 2 — GAN condicional: arquitetura, treinamento e evolução

Este é um **novo modelo condicional**; os scripts antigos e a interface continuam incondicionais. Os checkpoints antigos são incompatíveis. Não os sobrescreva. A condição entra nas duas redes, conforme [Mirza e Osindero (2014)](https://arxiv.org/abs/1411.1784).

Por padrão fazemos somente um teste rápido com 200 imagens. Para produzir evidências de aprendizado, selecione `TESTE_RAPIDO=False`, um novo nome e o número desejado de épocas. Reinicie o kernel e execute tudo em ordem.

In [ ]:
from pathlib import Path
import json, csv, time
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Funciona abrindo o Jupyter na raiz ou na pasta notebooks.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modelos.py").exists())
CLASSES = ['Camiseta/Top', 'Calça', 'Pulôver', 'Vestido', 'Casaco',
           'Sandália', 'Camisa', 'Tênis', 'Bolsa', 'Bota']
SEMENTE = 42
torch.set_num_threads(2)
torch.manual_seed(SEMENTE)
np.random.seed(SEMENTE)
DISPOSITIVO = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Raiz:', RAIZ, '| Dispositivo:', DISPOSITIVO)


In [ ]:
transformacao = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
base = datasets.FashionMNIST(str(RAIZ / 'data'), train=True, download=True, transform=transformacao)
teste = datasets.FashionMNIST(str(RAIZ / 'data'), train=False, download=True, transform=transformacao)
idx_treino, idx_validacao = train_test_split(
    np.arange(len(base)), train_size=50000, test_size=10000,
    stratify=base.targets.numpy(), random_state=SEMENTE)
treino, validacao = Subset(base, idx_treino), Subset(base, idx_validacao)
assert not set(idx_treino).intersection(idx_validacao)
assert len(treino) == 50000 and len(validacao) == len(teste) == 10000
print('Treino, validação, teste:', len(treino), len(validacao), len(teste))
print('Treino por classe:', np.bincount(base.targets[idx_treino]))


## Configuração da experiência
Dê um nome diferente a cada execução. Para continuar, use o mesmo nome, parâmetros e `RETOMAR=True`; `EPOCAS` é o total desejado. Não misture esse teste com o treinamento completo.

In [ ]:
# O padrão verifica o fluxo rapidamente. Não demonstra aprendizado de qualidade.
TESTE_RAPIDO = True
NOME_EXECUCAO = 'verificacao_cgan_notebook'
EPOCAS = 2  # total desejado, inclusive na retomada
RETOMAR = False
BATCH_SIZE = 128
assert EPOCAS >= 1 and BATCH_SIZE >= 1
assert NOME_EXECUCAO and all(c.isalnum() or c in '_-' for c in NOME_EXECUCAO)
PASTA = RAIZ / 'artefatos' / 'cgan' / NOME_EXECUCAO
MODELOS = RAIZ / 'checkpoints' / 'cgan' / NOME_EXECUCAO
CHECKPOINT = MODELOS / 'ultimo.pt'
if RETOMAR:
    assert CHECKPOINT.exists(), 'Não existe checkpoint para retomar.'
else:
    assert not PASTA.exists() and not MODELOS.exists(), 'Escolha outro nome ou use RETOMAR=True.'

# Subconjunto balanceado apenas para o teste de execução: 20 imagens por classe.
if TESTE_RAPIDO:
    rotulos_treino = base.targets[idx_treino].numpy()
    pequenos = np.concatenate([np.flatnonzero(rotulos_treino == c)[:20] for c in range(10)])
    treino = Subset(treino, pequenos.tolist())
config = dict(arquitetura='cgan_conv_v1', semente=SEMENTE, batch_size=BATCH_SIZE,
              teste_rapido=TESTE_RAPIDO, imagens_treino=len(treino), lr=0.0002)
print(config)


## Arquitetura condicional
O gerador concatena o ruído a uma representação aprendida da classe (embedding). O discriminador recebe dois canais: imagem e representação espacial da classe. Assim, os rótulos participam do aprendizado nas duas redes. Usamos uma arquitetura convolucional compacta para esta nova experiência; diferenças em relação ao modelo antigo impedem atribuir diferenças visuais apenas ao condicionamento.

In [ ]:
# A mesma definição é incluída no notebook 03 para permitir leitura independente.
DIM_RUIDO = 100
class GeradorCondicional(nn.Module):
    def __init__(self):
        super().__init__()
        self.classe = nn.Embedding(10, 10)
        self.rede = nn.Sequential(
            nn.ConvTranspose2d(DIM_RUIDO + 10, 128, 7, 1, 0, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 1, 4, 2, 1, bias=False), nn.Tanh())

    def forward(self, ruido, rotulos):
        condicao = self.classe(rotulos).view(-1, 10, 1, 1)
        return self.rede(torch.cat([ruido, condicao], dim=1))

class DiscriminadorCondicional(nn.Module):
    def __init__(self):
        super().__init__()
        self.classe = nn.Embedding(10, 28 * 28)
        self.rede = nn.Sequential(
            nn.Conv2d(2, 64, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 1, 7, 1, 0, bias=False))

    def forward(self, imagens, rotulos):
        mapa_classe = self.classe(rotulos).view(-1, 1, 28, 28)
        return self.rede(torch.cat([imagens, mapa_classe], dim=1))


## Entradas fixas e retomada
Fixamos ruído **e classe** para comparar épocas. Cada linha do painel é uma categoria e cada coluna reutiliza o mesmo ruído. As grades antigas não pertencem a esta sequência.

In [ ]:
gerador = GeradorCondicional().to(DISPOSITIVO)
discriminador = DiscriminadorCondicional().to(DISPOSITIVO)
otim_g = torch.optim.Adam(gerador.parameters(), lr=config['lr'], betas=(0.5, 0.999))
otim_d = torch.optim.Adam(discriminador.parameters(), lr=config['lr'], betas=(0.5, 0.999))
criterio = nn.BCEWithLogitsLoss()  # D retorna logits, sem Sigmoid na rede.
rng_loader = torch.Generator().manual_seed(SEMENTE)
# As mesmas três entradas são repetidas para cada classe.
ruido_fixo = torch.randn(3, DIM_RUIDO, 1, 1, device=DISPOSITIVO).repeat(10, 1, 1, 1)
rotulos_fixos = torch.arange(10, device=DISPOSITIVO).repeat_interleave(3)
historico, inicio = [], 1
if RETOMAR:
    estado = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
    assert estado['config'] == config, 'Configuração incompatível com a execução salva.'
    gerador.load_state_dict(estado['gerador']); discriminador.load_state_dict(estado['discriminador'])
    otim_g.load_state_dict(estado['otim_g']); otim_d.load_state_dict(estado['otim_d'])
    ruido_fixo = estado['ruido_fixo'].to(DISPOSITIVO)
    rotulos_fixos = estado['rotulos_fixos'].to(DISPOSITIVO)
    historico, inicio = estado['historico'], estado['epoca'] + 1
    torch.set_rng_state(estado['rng']); rng_loader.set_state(estado['rng_loader'])
    if DISPOSITIVO.type == 'cuda' and estado['rng_cuda'] is not None:
        torch.cuda.set_rng_state_all(estado['rng_cuda'])
loader = DataLoader(treino, batch_size=BATCH_SIZE, shuffle=True, generator=rng_loader)
PASTA.mkdir(parents=True, exist_ok=True); MODELOS.mkdir(parents=True, exist_ok=True)
(PASTA / 'configuracao.json').write_text(json.dumps(config, indent=2))


In [ ]:
def painel(imagens, titulo, caminho=None):
    # Dez linhas: uma por classe. Três colunas: três ruídos fixos.
    fig, eixos = plt.subplots(10, 3, figsize=(6, 15))
    for i, eixo in enumerate(eixos.flat):
        eixo.imshow(imagens[i, 0], cmap='gray', vmin=0, vmax=1)
        eixo.set_xticks([]); eixo.set_yticks([])
        if i % 3 == 0:
            eixo.set_ylabel(CLASSES[i // 3], rotation=0, ha='right', va='center')
    fig.suptitle(titulo)
    fig.tight_layout(rect=(0.15, 0, 1, 0.97))
    if caminho is not None:
        fig.savefig(caminho, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(fig)


In [ ]:
def salvar_torch(objeto, destino):
    temporario = destino.with_suffix('.tmp')
    torch.save(objeto, temporario)
    temporario.replace(destino)

def registrar_epoca(epoca):
    gerador.eval()
    with torch.no_grad():
        amostras = ((gerador(ruido_fixo, rotulos_fixos) + 1) / 2).clamp(0, 1).cpu()
    gerador.train()
    salvar_torch(amostras, MODELOS / f'amostras_{epoca:03d}.pt')
    salvar_torch(dict(arquitetura='cgan_conv_v1', epoca=epoca, config=config,
                      gerador=gerador.state_dict()), MODELOS / f'gerador_{epoca:03d}.pt')
    painel(amostras, f"cGAN — época {epoca} — {'TESTE DE EXECUÇÃO' if TESTE_RAPIDO else 'treino completo'}",
           PASTA / f'epoca_{epoca:03d}.png')

if inicio == 1:
    registrar_epoca(0)


## Treinamento alternado
D aprende com pares (imagem, classe); G recebe ruído e classe. A perda é calculada em logits para estabilidade numérica. Registramos a média ponderada por imagens em cada época, tempo, amostras e pesos. A existência de rótulos não garante qualidade ou correspondência visual: isso exige observar um treinamento suficiente.

In [ ]:
# Interromper durante uma época permite retomar a última época concluída.
# Se ainda não há checkpoint, escolha novo nome para recomeçar.
for epoca in range(inicio, EPOCAS + 1):
    gerador.train(); discriminador.train()
    t0 = time.perf_counter()
    soma_g = soma_d = quantidade = 0
    for lote, (imagens, classes) in enumerate(loader, 1):
        imagens, classes = imagens.to(DISPOSITIVO), classes.to(DISPOSITIVO)
        n = imagens.size(0)
        reais = torch.ones(n, 1, 1, 1, device=DISPOSITIVO)
        falsas = torch.zeros_like(reais)
        # D recebe imagem E classe, para os exemplos reais e gerados.
        otim_d.zero_grad(set_to_none=True)
        ruido = torch.randn(n, DIM_RUIDO, 1, 1, device=DISPOSITIVO)
        geradas = gerador(ruido, classes)
        perda_d = criterio(discriminador(imagens, classes), reais) + criterio(discriminador(geradas.detach(), classes), falsas)
        perda_d.backward(); otim_d.step()
        # G também recebe a classe e tenta produzir uma imagem adequada à condição.
        discriminador.requires_grad_(False)
        otim_g.zero_grad(set_to_none=True)
        perda_g = criterio(discriminador(geradas, classes), reais)
        perda_g.backward(); otim_g.step()
        discriminador.requires_grad_(True)
        assert torch.isfinite(perda_d) and torch.isfinite(perda_g)
        soma_d += perda_d.item() * n; soma_g += perda_g.item() * n; quantidade += n
        if lote % 50 == 0:
            print(f'Época {epoca}: lote {lote}/{len(loader)}')
    historico.append(dict(epoca=epoca, perda_d=soma_d/quantidade, perda_g=soma_g/quantidade,
                          segundos=time.perf_counter()-t0))
    salvar_torch(dict(epoca=epoca, config=config, gerador=gerador.state_dict(),
                     discriminador=discriminador.state_dict(), otim_g=otim_g.state_dict(),
                     otim_d=otim_d.state_dict(), ruido_fixo=ruido_fixo.cpu(), rotulos_fixos=rotulos_fixos.cpu(),
                     historico=historico, rng=torch.get_rng_state(), rng_loader=rng_loader.get_state(),
                     rng_cuda=torch.cuda.get_rng_state_all() if DISPOSITIVO.type=='cuda' else None), CHECKPOINT)
    with (PASTA / 'historico.csv').open('w', newline='') as arquivo:
        writer = csv.DictWriter(arquivo, fieldnames=['epoca','perda_d','perda_g','segundos'])
        writer.writeheader(); writer.writerows(historico)
    registrar_epoca(epoca)
    print(historico[-1])
print('Última época concluída:', historico[-1]['epoca'] if historico else 0)


## Acompanhamento das perdas

In [ ]:
if historico:
    fig, eixo = plt.subplots(figsize=(7, 4))
    for chave in ['perda_d', 'perda_g']:
        eixo.plot([h['epoca'] for h in historico], [h[chave] for h in historico], marker='o', label=chave)
    eixo.set(xlabel='Época', ylabel='Perda média', title='Perdas não são acurácia')
    eixo.legend(); fig.tight_layout(); fig.savefig(PASTA / 'perdas.png'); plt.show(); plt.close(fig)


## Registro para a apostila
Use as configurações, o CSV e os painéis identificados. Registre quais classes ganharam contornos reconhecíveis, quais continuam ambíguas e se há repetição. Não confunda duas épocas do teste de execução com treinamento completo. Não trate uma queda da perda como prova de melhoria visual. O notebook 03 permite comparar três momentos e solicitar uma classe ao modelo salvo.